# Sense Check: Cleaned Hedged Panel

This notebook inspects the cleaned panel produced by `03_02c_clean_after_hedge.py`.
It focuses on coverage, missingness, and return distributions after winsorization.

In [1]:
from pathlib import Path
import pandas as pd

# Use the notebook folder as the base directory
notebook_dir = Path.cwd()
# Outputs live under the local outputs folder
out_dir = notebook_dir / "outputs"

# Resolve expected file path
clean_path = out_dir / "bond_panel_monthly_clean.parquet"
if not clean_path.exists():
    raise FileNotFoundError(
        f"Missing {clean_path}. Run 03_02c_clean_after_hedge.py first."
    )

# Load cleaned panel
clean_panel = pd.read_parquet(clean_path, engine="fastparquet")
# Preview a few rows
clean_panel.head()

,RIC,Trade Date,price,month_end,yield,price_return,treasury_ret,credit_excess_return
0,001055AY8=,2018-11-30,99.653,2018-11-30,4.772,0.021569,-0.000227,0.021795
1,001055AY8=,2018-12-31,102.415,2018-12-31,4.600,0.027716,0.000258,0.027458
2,001055AY8=,2019-01-31,106.537,2019-01-31,4.354,0.040248,0.000276,0.039972
3,001055AY8=,2019-02-28,106.299,2019-02-28,4.367,-0.002234,-0.000668,-0.001566
4,001055AY8=,2019-03-29,110.436,2019-03-31,4.132,0.038919,0.000428,0.038490


In [2]:
# Inspect column dtypes and non-null counts
clean_panel.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99103 entries, 0 to 99102
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   RIC                   99103 non-null  object        
 1   Trade Date            99103 non-null  datetime64[ns]
 2   price                 98990 non-null  float64       
 3   month_end             99103 non-null  datetime64[ns]
 4   yield                 98685 non-null  float64       
 5   price_return          99103 non-null  float64       
 6   treasury_ret          99103 non-null  float64       
 7   credit_excess_return  99103 non-null  float64       
dtypes: datetime64[ns](2), float64(5), object(1)
memory usage: 6.0+ MB


In [3]:
# Coverage: date range, unique bonds, and total rows
clean_panel["month_end"].min(), clean_panel["month_end"].max(), clean_panel["RIC"].nunique(), len(clean_panel)

(Timestamp('2004-07-31 00:00:00'),
 Timestamp('2025-09-30 00:00:00'),
 1720,
 99103)

In [4]:
# Missingness rates (top 10)
clean_panel.isna().mean().sort_values(ascending=False).head(10)

yield                   0.004218
price                   0.001140
Trade Date              0.000000
RIC                     0.000000
month_end               0.000000
price_return            0.000000
treasury_ret            0.000000
credit_excess_return    0.000000
dtype: float64

In [5]:
# Return diagnostics after winsorization
clean_panel["credit_excess_return"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])

count    99103.000000
mean        -0.001184
std          0.030253
min         -0.087656
1%          -0.087656
5%          -0.054417
50%         -0.000960
95%          0.050124
99%          0.095226
max          0.095227
Name: credit_excess_return, dtype: float64

## Summary (Cleaned Panel)

Based on the exported summary:
- **Rows:** ~99k monthly observations
- **Bonds (RICs):** ~1,720
- **Date range:** 2004-07 to 2025-09 (month-end)
- **Missingness:** very low (yield ~0.4%, price ~0.1%; returns complete)
- **Winsorization:** `credit_excess_return` is capped to roughly [-0.0877, 0.0952]

This is a stable cleaned panel suitable for IPCA estimation.